# Raport: Lista 1
**Autor:** Bartosz Kruszewski 
**Data:** 2026-10-09

In [ ]:
#| include: false

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns

Zdefiniujmy funkcje pomocniczne do wyliczania wag oraz metryk.

In [ ]:
def compute_theta4_weights(n):
    p = (np.arange(1, n + 1) - 0.5) / n
    z = stats.norm.ppf(p)
    w = stats.norm.pdf(z)
    return w / np.sum(w)


def compute_triangular_weights(n):
    w = np.arange(1, n + 1, dtype=float)
    w = np.minimum(w, n + 1 - w)
    return w / np.sum(w)


def calculate_metrics(estimates, true_val):
    bias = np.mean(estimates) - true_val
    var = np.var(estimates, ddof=1)
    mse = np.mean((estimates - true_val)**2)
    return bias, var, mse

Zdefinujmy estymatory zgodnie z wytycznymi z treści zadania.

In [ ]:
def est_mean(X):
    """(i) Średnia arytmetyczna."""
    return np.mean(X, axis=1)


def est_median(X):
    """(ii) Mediana."""
    return np.median(X, axis=1)


def est_weighted_custom(X):
    """(iii) Średnia ważona z własnymi wagami trójkątnymi."""
    n = X.shape[1]
    weights = compute_triangular_weights(n)
    return np.dot(X, weights)


def est_order_stats(X):
    """(iv) Średnia ważona statystyk pozycyjnych oparta na N(0,1)."""
    n = X.shape[1]
    X_sorted = np.sort(X, axis=1)
    weights = compute_theta4_weights(n)
    return np.dot(X_sorted, weights)


def est_harmonic_mean(X):
    """(v) Średnia harmoniczna: n / sum(1/X_i)."""
    n = X.shape[1]
    with np.errstate(divide='ignore', invalid='ignore'):
        inv_sum = np.sum(1.0 / X, axis=1)
        res = n / inv_sum
    return res


def est_power_mean_3(X):
    """(vi) Średnia potęgowa rzędu 3."""
    mean_cube = np.mean(X**3, axis=1)
    return np.sign(mean_cube) * np.abs(mean_cube)**(1.0 / 3.0)


def est_trimmed_mean(X, trim_ratio=0.10):
    """(vii) Estymator własny: średnia ucinana."""
    n = X.shape[1]
    k = int(trim_ratio * n)
    if k == 0:
        return np.mean(X, axis=1)
    X_sorted = np.sort(X, axis=1)
    return np.mean(X_sorted[:, k:n-k], axis=1)


def est_sample_variance(X):
    """(z4 i) Estymator wariancji S^2."""
    return np.var(X, axis=1, ddof=1)


def est_variance_around_median(X):
    """(z4 ii) Estymator wariancji wokół mediany."""
    n = X.shape[1]
    med = np.median(X, axis=1, keepdims=True)
    return np.sum((X - med)**2, axis=1) / (n - 1)


def est_median_squared_dev(X):
    """(z4 iii) Mediana kwadratów odchyleń od średniej."""
    mean = np.mean(X, axis=1, keepdims=True)
    sq_deviations = (X - mean)**2
    return np.median(sq_deviations, axis=1)

In [ ]:
#| include: false

def run_simulation(n, M, loc, scale, target_theta, dist_func, estimators, injection=None):
    # 1. Generowanie proby
    if injection is not None:
        X_base = dist_func(loc=loc, scale=scale, size=(M, n - 1))
        X_injected = np.full((M, 1), injection)
        X = np.hstack((X_base, X_injected))
    else:
        X = dist_func(loc=loc, scale=scale, size=(M, n))

    # 2. Obliczenia estymatorow
    estimators_dict = {f"t{i+1}: {est.__name__}": est(X) for i, est in enumerate(estimators)}
    
    # 3. Metryki wzgledem target_theta
    records = [(name, *calculate_metrics(vals, target_theta)) for name, vals in estimators_dict.items()]
    df_metrics = pd.DataFrame(records, columns=['Estymator', 'Obciążenie (Bias)', 'Wariancja', 'MSE'])

    print(
        f"Metrics (n={n}, loc={loc}, scale={scale}, target_theta={target_theta}"
        f"{f', injection={injection}' if injection is not None else ''}):"
    )
    display(df_metrics.round(5))

    # 4. Wykres
    plt.figure(figsize=(8, 4.5))
    ax = sns.boxplot(data=estimators_dict, showmeans=True)
    ax.axhline(target_theta, color='red', ls='--', label=rf'Prawdziwy parametr = {target_theta}')
    ax.set(
        title=rf"$n={n}, \mathrm{{loc}}={loc}, \mathrm{{scale}}={scale}, \theta={target_theta}$", 
        xlabel='Estymator', 
        ylabel='Wartość estymatora'
    )
    ax.tick_params(axis='x', rotation=15)
    ax.legend()
    plt.tight_layout()
    plt.show()

Wykresy dla zadania 1

In [ ]:
#| echo: false
#| output: true

for n in [20, 100]:
    for theta in [0, 4]:
        for sigma in [1, 2]:
            run_simulation(
                n=n, loc=theta, scale=sigma, target_theta=theta, M=10000,
                dist_func=np.random.normal, estimators=[
                    est_mean, est_median, est_weighted_custom,
                    est_order_stats, est_harmonic_mean,
                    est_power_mean_3, est_trimmed_mean
                ]
            )

Wykresy dla zadania 2

In [ ]:
#| echo: false
#| output: true

for dist_func in [np.random.normal, np.random.logistic, stats.cauchy.rvs]:
    for theta in [0, 4]:
        for sigma in [1, 2]:
            run_simulation(
                n=50, loc=theta, scale=sigma, target_theta=theta, M=10000,
                dist_func=dist_func, estimators=[est_mean, est_median]
            )

Wykresy dla zadania 3

In [ ]:
#| echo: false
#| output: true

M, N = 10000, 50

rows = []
for k in range(10, 101, 10):
    X_base = np.random.normal(loc=0, scale=1, size=(M, N - 1))
    X = np.hstack((X_base, np.full((M, 1), k)))

    estimators = {'Średnia': est_mean(X), 'Mediana': est_median(X)}

    for est_name, vals in estimators.items():
        for m_name, val in zip(
            ['Obciążenie (Bias)', 'Wariancja', 'MSE'],
            calculate_metrics(vals, theta)
        ):
            rows.append({'k': k, 'Estymator': est_name, 'Metryka': m_name, 'Wartość': val})

df = pd.DataFrame(rows)

g = sns.relplot(
    data=df, x='k', y='Wartość', hue='Estymator', style='Estymator',
    col='Metryka', kind='line', markers=True, dashes=False,
    facet_kws={'sharey': False}, height=4, aspect=1.1
)
g.set_axis_labels('Wartość k', 'Wartość metryki')
g.set_titles(col_template='{col_name}')
plt.show()

df.pivot(index='k', columns=['Metryka', 'Estymator'], values='Wartość').round(5)

Wykresy dla zadania 4

In [ ]:
#| echo: false
#| output: true

for mu in [0, 4]:
    for theta in [1, 4]:
        run_simulation(
            n=50, loc=mu, scale=np.sqrt(theta), target_theta=theta, M=10000,
            dist_func=np.random.normal, estimators=[
                est_sample_variance, est_variance_around_median, est_median_squared_dev
            ]
        )